In [7]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from peft import prepare_model_for_kbit_training
from collections import Counter
from transformers import BitsAndBytesConfig
from collections import Counter

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

In [8]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
            
        loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # predictions = (logits > 0).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "precision_macro": precision_score(labels, predictions, average="macro"),
        "recall_macro": recall_score(labels, predictions, average="macro")
    }

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [10]:
with open("data/regional-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Regional Court Data: {len(data)}")

Number of samples per label:
nevinovat    : 3626 samples
vinovat      : 5244 samples

Total samples in Regional Court Data: 8870


In [11]:
try:
    from huggingface_hub import login
    login(YOUR_HF_TOKEN)
except Exception as e:
    print("Hugging Face login skipped:", e)

In [6]:
from evaluate import load

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
# sacrebleu = evaluate.load("sacrebleu")
# bertscore = load("bertscore")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    results = clf_metrics.compute(predictions=predictions, references=labels)
    
    decoded_preds = [str(p) for p in predictions]
    decoded_labels = [str(l) for l in labels]
    
    results.update(rouge.compute(predictions=decoded_preds, references=decoded_labels))
    results.update(meteor.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels]))
    # results.update(sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels]))
    # results.update(bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang="en"))
    
    return results

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bistreamt/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/bistreamt/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bistreamt/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Zero-SHOT

In [8]:
def prepare_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    label_map = {
        "vinovat": 0,
        "nevinovat": 1
    }

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        
        # --- PROMPT ENGINEERING START ---
        
        just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        instructional_prompt = (
            f"Decide verdictul acestui caz juridic:\n"
            f"Ai de ales între două răspunsuri: vinovat sau nevinovat.\n"
            f"DESCRIERE: {desc}\n"
            f"JUSTIFICARE: {just}\n"
        )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })
    
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset("data/regional-court-data.json")

## Llama

In [9]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.28s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,2.265600,1.115914,0.512397,0.415842,0.407407,0.424632,0.511645,0.000000,0.513148,0.512397,0.256198
300,2.384300,1.132706,0.575507,0.267185,0.453744,0.189338,0.575507,0.000000,0.576258,0.575507,0.287754
450,2.302800,0.920326,0.586777,0.335749,0.489437,0.255515,0.586777,0.000000,0.587528,0.587528,0.293388
600,1.865300,0.826404,0.553719,0.453039,0.453875,0.452206,0.554470,0.000000,0.553719,0.553719,0.276860
750,1.754600,0.887898,0.607062,0.238719,0.573427,0.150735,0.607062,0.000000,0.607814,0.607814,0.303531
900,1.737300,0.756482,0.622840,0.282857,0.634615,0.181985,0.622840,0.000000,0.622840,0.623591,0.311420
1050,1.467900,0.691777,0.644628,0.579556,0.561102,0.599265,0.643877,0.000000,0.645755,0.645379,0.322314
1200,1.739900,0.734520,0.682194,0.475836,0.730038,0.352941,0.682194,0.000000,0.682945,0.682194,0.341097
1350,1.675200,0.710016,0.716754,0.597652,0.712468,0.514706,0.717130,0.000000,0.717506,0.717506,0.358377
1500,2.307100,0.878058,0.691961,0.483627,0.768000,0.352941,0.691961,0.000000,0.691961,0.692712,0.345980


KeyboardInterrupt: 

## RoLlama

In [9]:
model_name = "OpenLLM-Ro/RoLlama2-7b-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:17<00:00,  5.68s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoLlama2-7b-Base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,2.144700,0.986623,0.557476,0.427600,0.453608,0.404412,0.556724,0.000000,0.557476,0.556724,0.278738
300,2.052800,0.988571,0.580015,0.322424,0.473310,0.244485,0.580015,0.000000,0.580766,0.580766,0.290008
450,2.520200,1.585165,0.595041,0.059337,0.586207,0.031250,0.595041,0.000000,0.594290,0.595041,0.297521
600,2.209900,0.885617,0.567243,0.355705,0.454286,0.292279,0.567243,0.000000,0.566491,0.567243,0.283621
750,1.783000,1.085568,0.604057,0.180404,0.585859,0.106618,0.604057,0.000000,0.603306,0.604808,0.302029
900,1.735600,0.868106,0.604057,0.288799,0.543147,0.196691,0.604057,0.000000,0.604057,0.604808,0.302029
1050,1.783000,0.768870,0.601052,0.473736,0.513978,0.439338,0.601052,0.000000,0.601052,0.601803,0.300526


KeyboardInterrupt: 

## RoGemma

In [9]:
model_name = "OpenLLM-Ro/RoGemma-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00,  3.95s/it]
Some weights of GemmaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoGemma-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,5.404800,1.358766,0.566491,0.279650,0.435798,0.205882,0.566491,0.000000,0.566491,0.564989,0.283246
300,2.616600,1.464984,0.586777,0.337349,0.489510,0.257353,0.586777,0.000000,0.587528,0.586026,0.293388
450,3.041400,1.346075,0.598798,0.185976,0.544643,0.112132,0.598798,0.000000,0.598047,0.598798,0.299399
600,2.140400,1.004643,0.583772,0.324390,0.481884,0.244485,0.583020,0.000000,0.583772,0.583772,0.291886
750,2.046800,1.234943,0.595793,0.129450,0.540541,0.073529,0.595793,0.000000,0.595793,0.595793,0.297896
900,1.951700,0.803572,0.578512,0.419855,0.479905,0.373162,0.579264,0.000000,0.579264,0.578512,0.289256
1050,1.836600,1.028681,0.606311,0.222552,0.576923,0.137868,0.605560,0.000000,0.605935,0.606311,0.303156


KeyboardInterrupt: 

## RoMistral

In [9]:
model_name = "OpenLLM-Ro/RoMistral-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:11<00:00,  3.80s/it]
Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoMistral-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,6.827800,2.682696,0.519910,0.418562,0.414414,0.422794,0.519910,0.000000,0.519910,0.519159,0.259955
300,5.256600,2.280426,0.564989,0.468320,0.467890,0.468750,0.564989,0.000000,0.564989,0.564237,0.282494
450,4.828200,1.902198,0.582269,0.198847,0.460000,0.126838,0.581518,0.000000,0.580766,0.582269,0.291134
600,3.357000,2.464516,0.592036,0.021622,0.545455,0.011029,0.591285,0.000000,0.591285,0.592036,0.296018
750,2.859400,1.911992,0.606311,0.215569,0.580645,0.132353,0.605560,0.000000,0.604808,0.606311,0.303156
900,2.853400,0.817511,0.613824,0.400932,0.547771,0.316176,0.613073,0.000000,0.613824,0.613824,0.306912
1050,2.573800,0.782177,0.600301,0.532513,0.510101,0.556985,0.601052,0.000000,0.600301,0.601052,0.300150
1200,2.118900,1.643918,0.591285,0.000000,0.000000,0.000000,0.590533,0.000000,0.590533,0.591285,0.295642
1350,2.445500,0.800962,0.671675,0.603808,0.595707,0.612132,0.671675,0.000000,0.671675,0.671675,0.335838
1500,2.473500,0.961284,0.691961,0.623162,0.623162,0.623162,0.692712,0.000000,0.692712,0.692337,0.345980


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


KeyboardInterrupt: 

## RoGPT2

In [10]:
model_name = "readerbench/RoGPT2-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Important for GPT2 padding
model.config.pad_token_id = tokenizer.pad_token_id

model = model.to(device)

print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/regional-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Map: 100%|██████████| 1331/1331 [00:01<00:00, 1283.43 examples/s]
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at readerbench/RoGPT2-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
50,0.730900,0.733366,0.507889,0.345654,0.378556,0.318015,0.507137,0.000000,0.508264,0.507137,0.253944
100,0.708000,0.705125,0.555222,0.348018,0.434066,0.290441,0.555222,0.000000,0.555973,0.554470,0.277611
150,0.675500,0.684056,0.578512,0.470255,0.483495,0.457721,0.578512,0.000000,0.579264,0.577010,0.289256
200,0.647100,0.659726,0.622840,0.467091,0.552764,0.404412,0.623591,0.000000,0.623591,0.622840,0.311420
250,0.632700,0.640191,0.637115,0.567592,0.553229,0.582721,0.637866,0.000000,0.637115,0.637115,0.318557
300,0.603800,0.629041,0.642374,0.635528,0.544619,0.762868,0.643125,0.000000,0.643125,0.642374,0.321187
350,0.568500,0.561341,0.713749,0.679563,0.626357,0.742647,0.713749,0.000000,0.712998,0.713749,0.356875
400,0.566600,0.527862,0.721262,0.664253,0.654189,0.674632,0.722014,0.000000,0.721262,0.721262,0.360631
450,0.547300,0.567218,0.718257,0.535316,0.821293,0.397059,0.719008,0.000000,0.717506,0.718257,0.359128
500,0.501000,0.491197,0.754320,0.702457,0.695495,0.709559,0.755071,0.000000,0.753569,0.754320,0.377160


KeyboardInterrupt: 

# Chain-of-Thought Prompting

In [ ]:
def prepare_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    label_map = {
        "vinovat": 0,
        "nevinovat": 1
    }

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()

        # --- PROMPT ENGINEERING START ---
        instructional_prompt = (
            f"Ești un expert legal și task-ul tău este de a analiza "
            f"un caz juridic și de a decide verdictul corect. "
            f"Analizează pas cu pas informațiile oferite înainte "
            f"de a formula răspunsul final. "
            f"Decide verdictul acestui caz juridic:\n"
            f"Ai de ales între două răspunsuri: admis sau respins.\n"
            f"Descriere caz: {desc}\n" 
        )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })

    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset("data/regional-court-data.json")

## Llama

In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

## RoLlama

In [ ]:
model_name = "OpenLLM-Ro/RoLlama2-7b-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

## RoGemma

In [ ]:
model_name = "OpenLLM-Ro/RoGemma-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

## RoMistral

In [ ]:
model_name = "OpenLLM-Ro/RoMistral-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

## RoGPT2

In [ ]:
model_name = "readerbench/RoGPT2-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Important for GPT2 padding
model.config.pad_token_id = tokenizer.pad_token_id

model = model.to(device)

print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/regional-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()